# 0.10 — GenAI hierarchy drill-down (stage 2)

**Prerequisite:** run [`0.9`](0.9-genai-unsupervised-detection.ipynb) stage-1 all-news BERTrend → `genai_stage1_parents_{21,28}d.parquet`.

Pick persistent **macro parents** from stage 1, scope the **full** Bloomberg corpus, re-run BERTrend with finer clustering:

1. **Exp A — embedding-scoped:** `cos(headline, parent_centroid) ≥ τ`
2. **Exp B — keyword-scoped:** match ≥1 discriminative parent keyword

**Gate:** `first_seen < 2023-05-17` (CHAT inception) · qualitative genAI read on keywords + rep headlines.

**Outputs:** `scan_genai_embed_P{id}_{21}d.parquet`, `scan_genai_kw_P{id}_{21}d.parquet`, `genai_drilldown_compare.parquet`

In [1]:
import os, sys, lzma, re
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
from loguru import logger as _lg
_lg.remove(); _lg.add(sys.stderr, level="WARNING")

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("BERTREND_BASE_DIR", str(_ROOT / "notebooks" / "output" / "bertrend_base"))
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.representation import MaximalMarginalRelevance
from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN, SOURCE_COLUMN, TEXT_COLUMN, TIMESTAMP_COLUMN, URL_COLUMN, group_by_days,
)

# --- config (align with 0.9) ---
YEARS = [2021, 2022, 2023]
DATE_START, DATE_END = pd.Timestamp("2021-01-01"), pd.Timestamp("2023-12-31")
INCEPTION = pd.Timestamp("2023-05-17")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42

STAGE1_G = 21                              # granularity of stage-1 parents to load
STAGE1_THEMES = OUTPUT_DIR / f"scan_genai_allnews_{STAGE1_G}d.parquet"
STAGE1_PARENTS = OUTPUT_DIR / f"genai_stage1_parents_{STAGE1_G}d.parquet"

DRILL_GRANULARITY = 21
DRILL_MIN_TOPIC, DRILL_MIN_SAMPLES = 8, 3
DRILL_MIN_SIM = 0.65
VECTORIZER_MIN_DF = 1
MIN_ACTIVE_SLICES = 4

# Parent selection: explicit IDs override auto-pick
PARENT_IDS = None                          # e.g. [1, 9, 21] — None = auto from PARENT_HINT_RE
PARENT_HINT_RE = re.compile(
    r"technology|technologies|software|digital|data|chip|semiconductor|"
    r"cloud|computing|cyber|intel|nvidia|artificial",
    re.I,
)
MAX_PARENTS = 4
MIN_PARENT_SLICES = 4

# Scoping thresholds
EMBED_THRESH = 0.55                        # tune per parent if subcorpus too big/small
SCOPE_CAP = 120_000                        # max headlines per scoped run (stratified by day)
KEYWORD_TOP_K = 5                          # discriminative keywords for Exp B

EMB_CACHE = OUTPUT_DIR / "genai_full_emb_2021_2023.npy"
META_CACHE = OUTPUT_DIR / "genai_full_meta.parquet"

print(f"Device {DEVICE} | stage-1 parents: {STAGE1_PARENTS.name}")

Device mps | stage-1 parents: genai_stage1_parents_21d.parquet


## 1. Load Bloomberg headlines (full corpus)

In [2]:
def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text

frames = []
for yr in YEARS:
    with lzma.open(RAW_DIR / f"raw_news_{yr}.csv.xz", "rb") as f:
        part = (
            pl.scan_csv(f, infer_schema_length=10_000)
            .select(["Headline", "CaptureTime", "WireName"])
            .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
            .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
            .collect()
        )
    frames.append(part)
    print(f"{yr}: {part.height:>9,} Bloomberg rows")

news = pl.concat(frames).to_pandas()
news["date"] = pd.to_datetime(news["CaptureTime"]).dt.tz_localize(None)
news = news[(news.date >= DATE_START) & (news.date <= DATE_END)]
news = news.dropna(subset=["Headline"]).drop_duplicates("Headline")
news["Headline"] = news["Headline"].map(strip_prefix)
news = news[news.Headline.str.split().map(len) >= 4].reset_index(drop=True)
print(f"\nFull corpus: {len(news):,} headlines")

2021: 5,889,523 Bloomberg rows
2022: 5,670,511 Bloomberg rows
2023: 5,557,627 Bloomberg rows

Full corpus: 2,954,113 headlines


## 2. Helpers + stage-1 parent load

In [3]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

CUSTOM_STOP = list(ENGLISH_STOP_WORDS.union({
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}))
GENERIC_KW = set(CUSTOM_STOP) | {
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "sales", "deal", "chief", "cut", "raised", "raise", "buy", "sell", "net", "revenue",
    "beat", "miss", "forecast", "outlook", "business", "market", "markets", "global",
    "first", "second", "third", "fourth", "annual", "meeting", "plans", "plan",
}

def make_df(sub: pd.DataFrame) -> pd.DataFrame:
    d = pd.DataFrame({TEXT_COLUMN: sub["Headline"].values,
                      TIMESTAMP_COLUMN: pd.to_datetime(sub["date"].values)})
    d[DOCUMENT_ID_COLUMN] = range(len(d))
    d[SOURCE_COLUMN] = "bloomberg"
    d[URL_COLUMN] = None
    return d.reset_index(drop=True)

def embed(texts_or_df, show_progress=True) -> np.ndarray:
    texts = texts_or_df[TEXT_COLUMN].tolist() if isinstance(texts_or_df, pd.DataFrame) else texts_or_df
    return embedder.encode(texts, batch_size=64, show_progress_bar=show_progress,
                           convert_to_numpy=True, normalize_embeddings=True)

def _bertopic(min_topic_size, min_samples, min_df=VECTORIZER_MIN_DF, cluster_method="eom"):
    cfg = f"""
[global]
language = "English"
[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = []
zeroshot_min_similarity = 0
[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}
[hdbscan_model]
min_cluster_size = {min_topic_size}
min_samples = {min_samples}
metric = "euclidean"
cluster_selection_method = "{cluster_method}"
prediction_data = true
[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = {min_df}
[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true
[mmr_model]
diversity = 0.3
[reduce_outliers]
strategy = "c-tf-idf"
"""
    tm = BERTopicModel(cfg)
    tm.vectorizer_model = CountVectorizer(stop_words=CUSTOM_STOP,
                                          token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
                                          ngram_range=(1, 2), min_df=min_df)
    tm.config["bertopic_model"]["representation_model"] = [MaximalMarginalRelevance(diversity=0.4)]
    return tm

def run_bertrend(df, embeddings, granularity, min_topic_size, min_samples,
                 min_similarity=DRILL_MIN_SIM, tmp_tag="drill"):
    bt = BERTrend(topic_model=_bertopic(min_topic_size, min_samples))
    bt.config["granularity"] = granularity
    bt.config["min_similarity"] = min_similarity
    grouped = {ts: g for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty}
    bt.train_topic_models(grouped_data=grouped, embedding_model=embedder, embeddings=embeddings,
                          bertrend_models_path=OUTPUT_DIR / f"_genai_{tmp_tag}", save_topic_models=False)
    if bt.merged_df is None:
        return None
    bt.calculate_signal_popularity()
    return bt

def theme_table(bt) -> pd.DataFrame:
    rep = {}
    for _, r in bt.merged_df.drop_duplicates("Topic").iterrows():
        x = r.get("Representation")
        rep[int(r["Topic"])] = ", ".join(x[:8]) if isinstance(x, (list, tuple)) else str(x)
    rows = []
    for tid, d in bt.topic_sizes.items():
        st = pd.to_datetime(list(d.get("Timestamps", [])))
        if len(st) == 0:
            continue
        rows.append({"theme_id": int(tid), "slices": int(st.normalize().nunique()),
                     "first_seen": st.min().normalize(), "last_seen": st.max().normalize(),
                     "docs": int(max(d.get("Docs_Count", [0]) or [0])),
                     "keywords": rep.get(int(tid), "")})
    return pd.DataFrame(rows).sort_values(["slices", "docs"], ascending=False).reset_index(drop=True)

def rep_headlines(centroid, df, emb, n=5) -> list[str]:
    c = np.asarray(centroid, dtype=float)
    c = c / (np.linalg.norm(c) + 1e-12)
    idx = np.argsort(-(emb @ c))[:n]
    return df.iloc[idx][TEXT_COLUMN].tolist()

def discriminative_keywords(keywords: str, top_k: int = KEYWORD_TOP_K) -> list[str]:
    words = [w.strip().lower() for w in keywords.split(",") if w.strip()]
    out = [w for w in words if w not in GENERIC_KW and len(w) >= 3]
    return out[:top_k]

def _as_centroid(row) -> np.ndarray:
    e = row["embedding"]
    v = np.asarray(e if isinstance(e, np.ndarray) else list(e), dtype=float)
    return v / (np.linalg.norm(v) + 1e-12)

def cap_stratified(sub: pd.DataFrame, cap: int) -> pd.DataFrame:
    if len(sub) <= cap:
        return sub
    per_day = max(1, cap // sub["date"].dt.normalize().nunique())
    return (sub.groupby(sub["date"].dt.normalize(), group_keys=False)
              .apply(lambda g: g.sample(min(len(g), per_day), random_state=RANDOM_SEED)))

def print_themes(label, t, n_slices, parent_id, method):
    pre = t[(t.slices >= MIN_ACTIVE_SLICES) & (t.first_seen < INCEPTION)]
    print(f"\n{'=' * 72}\n{label} · parent T{parent_id} · {method} · {len(t)} subthemes · {n_slices} slices")
    print(f"Pre-CHAT stable: {len(pre)}")
    for _, r in t.head(10).iterrows():
        mark = "✓" if r.first_seen < INCEPTION else " "
        flag = "★" if r.slices >= MIN_ACTIVE_SLICES else " "
        print(f"  [{mark}{flag}] T{int(r.theme_id):>3} {int(r.slices):>2} slices  "
              f"{r['first_seen'].date()}  {r.keywords[:52]}")
    return pre

# --- load stage-1 ---
if not STAGE1_PARENTS.exists():
    raise FileNotFoundError(f"Run 0.9 first — missing {STAGE1_PARENTS}")
parents = pd.read_parquet(STAGE1_PARENTS)
parents["first_seen"] = pd.to_datetime(parents["first_seen"])
print(f"Stage-1 parents: {len(parents)} themes @ {STAGE1_G}d")
print(parents[["theme_id", "slices", "keywords"]].head(8).to_string(index=False))

if PARENT_IDS is not None:
    parent_rows = parents[parents.theme_id.isin(PARENT_IDS)].copy()
else:
    hint = parents.keywords.str.contains(PARENT_HINT_RE, na=False)
    parent_rows = (parents[hint & (parents.slices >= MIN_PARENT_SLICES)]
                   .sort_values(["slices", "docs"], ascending=False)
                   .head(MAX_PARENTS))
if parent_rows.empty:
    raise RuntimeError("No parent themes matched — set PARENT_IDS manually.")
print(f"\nDrill-down parents: {parent_rows.theme_id.tolist()}")
for _, r in parent_rows.iterrows():
    print(f"  T{int(r.theme_id):>3}  {int(r.slices):>2} slices  {r.keywords[:60]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Stage-1 parents: 86 themes @ 21d
 theme_id  slices                                                                                          keywords
        0      53                                   dollar, inflation, inside, yields, markets, steady, gold, rates
       34      53                florida, sba backs, florida sba, sba, jan florida, backs, proposals, proposals jan
       27      53            calstrs, calstrs backs, egm calstrs, proposals, backs, proposals jan, agm calstrs, egm
       20      53                            cut hold, cut, hold, cut sell, cut neutral, euros, neutral, cut reduce
       13      53 rated, rated buy, overweight, rated outperform, stanley, morgan stanley, morgan, rated overweight
        4      53                                       names, officer, appoints, hires, chief, sky, chairman, role
       11      53 premarket, shell, weekend, origin, tinto astrazeneca, icahn, equities wrap, industrials premarket
       12      53                 raise

## 3. Embed full corpus (cached)

One-time encode of all headlines — reused for every parent scope.

In [4]:
if EMB_CACHE.exists() and META_CACHE.exists():
    emb_full = np.load(EMB_CACHE, mmap_mode="r")
    meta = pd.read_parquet(META_CACHE).reset_index(drop=True)
    assert len(meta) == emb_full.shape[0]
    print(f"Loaded cache: {emb_full.shape}  {META_CACHE.name}")
else:
    meta = news[["Headline", "date"]].copy().reset_index(drop=True)
    print(f"Embedding full corpus ({len(meta):,}) — slow step…")
    emb_full = embed(make_df(meta))
    np.save(EMB_CACHE, emb_full)
    meta.to_parquet(META_CACHE, index=False)
    print(f"Wrote {EMB_CACHE.name} + {META_CACHE.name}")

Loaded cache: (2954113, 768)  genai_full_meta.parquet


## 4. Exp A — embedding-scoped BERTrend

In [5]:
embed_results = {}

for _, prow in parent_rows.iterrows():
    pid = int(prow["theme_id"])
    centroid = _as_centroid(prow)
    sims = emb_full @ centroid
    mask = sims >= EMBED_THRESH
    sub_news = cap_stratified(meta.loc[mask].copy(), SCOPE_CAP).sort_values("date")
    print(f"\nParent T{pid}: embed≥{EMBED_THRESH} → {mask.sum():,} hits, using {len(sub_news):,}")
    if len(sub_news) < 500:
        print("  skip — subcorpus too small")
        continue
    df_sub = make_df(sub_news)
    emb_sub = np.asarray(emb_full[sub_news.index.to_numpy()])
    bt = run_bertrend(df_sub, emb_sub, DRILL_GRANULARITY, DRILL_MIN_TOPIC, DRILL_MIN_SAMPLES,
                      tmp_tag=f"embed_P{pid}")
    if bt is None:
        print("  no merged subthemes")
        continue
    t = theme_table(bt)
    n_sl = sum(1 for gg in group_by_days(df_sub, DRILL_GRANULARITY).values() if not gg.empty)
    pre = print_themes(f"EMBED-SCOPED", t, n_sl, pid, f"τ={EMBED_THRESH}")
    out = t.assign(parent_id=pid, method="embed", embed_thresh=EMBED_THRESH,
                   scope_n=len(sub_news), granularity_days=DRILL_GRANULARITY)
    path = OUTPUT_DIR / f"scan_genai_embed_P{pid}_{DRILL_GRANULARITY}d.parquet"
    out.to_parquet(path, index=False)
    embed_results[pid] = {"bt": bt, "table": t, "pre": pre, "path": path, "df": df_sub, "emb": emb_sub}
    print(f"  → {path.name}")

KeyboardInterrupt: 

## 5. Exp B — keyword-scoped BERTrend

In [6]:
kw_results = {}

for _, prow in parent_rows.iterrows():
    pid = int(prow["theme_id"])
    kws = discriminative_keywords(str(prow["keywords"]))
    if not kws:
        print(f"\nParent T{pid}: no discriminative keywords — skip")
        continue
    pat = re.compile("|".join(re.escape(k) for k in kws), re.I)
    mask = meta["Headline"].str.contains(pat, na=False)
    sub_news = cap_stratified(meta.loc[mask].copy(), SCOPE_CAP).sort_values("date")
    print(f"\nParent T{pid}: keywords {kws} → {mask.sum():,} hits, using {len(sub_news):,}")
    if len(sub_news) < 500:
        print("  skip — subcorpus too small")
        continue
    df_sub = make_df(sub_news)
    emb_sub = np.asarray(emb_full[sub_news.index.to_numpy()])
    bt = run_bertrend(df_sub, emb_sub, DRILL_GRANULARITY, DRILL_MIN_TOPIC, DRILL_MIN_SAMPLES,
                      tmp_tag=f"kw_P{pid}")
    if bt is None:
        print("  no merged subthemes")
        continue
    t = theme_table(bt)
    n_sl = sum(1 for gg in group_by_days(df_sub, DRILL_GRANULARITY).values() if not gg.empty)
    pre = print_themes(f"KEYWORD-SCOPED", t, n_sl, pid, ",".join(kws[:3]))
    out = t.assign(parent_id=pid, method="keyword", scope_keywords=",".join(kws),
                   scope_n=len(sub_news), granularity_days=DRILL_GRANULARITY)
    path = OUTPUT_DIR / f"scan_genai_kw_P{pid}_{DRILL_GRANULARITY}d.parquet"
    out.to_parquet(path, index=False)
    kw_results[pid] = {"bt": bt, "table": t, "pre": pre, "path": path, "keywords": kws}
    print(f"  → {path.name}")


Parent T90: keywords ['demand remains', 'damage', 'check', 'cause real', 'real damage'] → 3,792 hits, using 3,792
2026-06-15 15:39:06.303 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 1/53...
2026-06-15 15:39:06.304 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2021-01-01 00:00:00
2026-06-15 15:39:06.304 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 104
2026-06-15 15:39:06.304 | DEBUG    | bertrend.BERTrend:_train_by_period:189 - Creating topic model...
2026-06-15 15:39:06.304 | DEBUG    | bertrend.BERTopicModel:fit:263 - 	Initializing BERTopic model
2026-06-15 15:39:06.306 | SUCCESS  | bertrend.BERTopicModel:fit:273 - 	BERTopic model instance created successfully
2026-06-15 15:39:06.306 | DEBUG    | bertrend.BERTopicModel:fit:275 - 	Fitting BERTopic model
2026-06-15 15:39:08.880 | DEBUG    | bertrend.BERTopicModel:fit:282 - 	Reducing outliers
2026-06-15 15:39:08.881 | DEBUG    | bertrend.BERTopicMo

2026-06-15 15:39:08,881 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 15:39:08.884 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 15:39:08.884 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 15:39:08.891 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2021-01-01 00:00:00
2026-06-15 15:39:08.891 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 2/53...
2026-06-15 15:39:08.891 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2021-01-22 00:00:00
2026-06-15 15:39:08.891 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 140
2026-06-15 15:39:08.892 | DEBUG    | bertrend.BERTrend:_train_by_period:189 - Creating topic model...
2026-06-15 15:39:08.892 | DEBUG    | bertrend.BERTopicModel:fit:263 - 	Initializing BERTopic model
2026-06-15 15:39:08.892 | SUCCESS  | bertrend.BERTopicModel:fit:273 - 	BERTopic model instance created successfully
2026-0

2026-06-15 15:39:08,999 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 15:39:09.002 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 15:39:09.003 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 15:39:09.007 | DEBUG    | bertrend.BERTrend:merge_models_with:360 - Merging topic models for timestamps: 2021-01-01 00:00:00 and 2021-01-22 00:00:00
2026-06-15 15:39:09.014 | SUCCESS  | bertrend.BERTrend:merge_models_with:428 - Models 2021-01-22 00:00:00 merged successfully with others
2026-06-15 15:39:09.014 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2021-01-22 00:00:00
2026-06-15 15:39:09.014 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 3/53...
2026-06-15 15:39:09.015 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2021-02-12 00:00:00
2026-06-15 15:39:09.015 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 138
2026-06-15 15:39:09.015 |

...

2026-06-15 15:39:12.537 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 15:39:12.537 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 15:39:12.543 | DEBUG    | bertrend.BERTrend:merge_models_with:360 - Merging topic models for timestamps: 2023-10-27 00:00:00 and 2023-11-17 00:00:00
2026-06-15 15:39:12.550 | SUCCESS  | bertrend.BERTrend:merge_models_with:428 - Models 2023-11-17 00:00:00 merged successfully with others
2026-06-15 15:39:12.551 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2023-11-17 00:00:00
2026-06-15 15:39:12.551 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 52/53...
2026-06-15 15:39:12.551 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2023-12-08 00:00:00
2026-06-15 15:39:12.551 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 43
2026-06-15 15:39:12.552 |

2026-06-15 15:39:12,589 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-15 15:39:12.592 | SUCCESS  | bertrend.BERTopicModel:fit:311 - 	BERTopic model fitted successfully
2026-06-15 15:39:12.592 | DEBUG    | bertrend.BERTrend:_train_by_period:195 - Topic model created successfully
2026-06-15 15:39:12.596 | DEBUG    | bertrend.BERTrend:merge_models_with:360 - Merging topic models for timestamps: 2023-11-17 00:00:00 and 2023-12-08 00:00:00
2026-06-15 15:39:12.600 | SUCCESS  | bertrend.BERTrend:merge_models_with:428 - Models 2023-12-08 00:00:00 merged successfully with others
2026-06-15 15:39:12.600 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2023-12-08 00:00:00
2026-06-15 15:39:12.601 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 53/53...
2026-06-15 15:39:12.601 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2023-12-29 00:00:00
2026-06-15 15:39:12.601 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 5
2026-06-15 15:39:12.601 | 

/Users/federicocinus/CODE - how/ThematicTrading/.venv/lib/python3.12/site-packages/umap/spectral.py:519: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(
/Users/federicocinus/CODE - how/ThematicTrading/.venv/lib/python3.12/site-packages/umap/spectral.py:519: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(


2026-06-15 15:39:12.638 | ERROR    | bertrend.BERTrend:train_topic_models:316 - Traceback:
Traceback (most recent call last):

  File "/Users/federicocinus/CODE - how/ThematicTrading/.venv/lib/python3.12/site-packages/bertopic/_bertopic.py", line 3944, in _reduce_dimensionality
    umap_embeddings = self.umap_model.fit_transform(embeddings, y=y)
                      │    │          │             │             └ None
                      │    │          │             └ array([[-0.06200092, -0.03331801,  0.05286498, ..., -0.05242934,
                      │    │          │                        0.03277235, -0.05146763],
                      │    │          │                      [-0.05153273,  0.0...
                      │    │          └ <function UMAP.fit_transform at 0x1389f6340>
                      │    └ UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=5, n_jobs=1, random_state=42, tqdm_kwds={'bar_format...
                      └ <bertopic._bertopic.B

## 6. Compare methods + rep headlines

Qualitative genAI check: read keywords + reps for pre-CHAT subthemes.

In [7]:
GENAI_HINT = re.compile(
    r"chatgpt|openai|generative|\\bllm\\b|artificial intelligence|nvidia|"
    r"gpu|copilot|anthropic|claude|bard|gemini|large language",
    re.I,
)

rows = []
for pid in parent_rows.theme_id.astype(int):
    for method, res_dict in [("embed", embed_results), ("keyword", kw_results)]:
        if pid not in res_dict:
            continue
        t = res_dict[pid]["table"]
        pre = t[(t.slices >= MIN_ACTIVE_SLICES) & (t.first_seen < INCEPTION)]
        genai_like = pre[pre.keywords.str.contains(GENAI_HINT, na=False)]
        rows.append({
            "parent_id": pid, "method": method, "n_subthemes": len(t),
            "pre_chat_stable": len(pre), "genai_keyword_hits": len(genai_like),
            "top_pre_chat": pre.iloc[0].keywords[:60] if len(pre) else "",
        })
compare = pd.DataFrame(rows)
if not compare.empty:
    compare.to_parquet(OUTPUT_DIR / "genai_drilldown_compare.parquet", index=False)
    print(compare.to_string(index=False))
else:
    print("No drill-down results to compare.")

print(f"\nMilestones: ChatGPT {CHATGPT_LAUNCH.date()} · CHAT {INCEPTION.date()}")
for pid, res in embed_results.items():
    if res["pre"].empty:
        continue
    best = res["pre"].sort_values(["slices", "first_seen"], ascending=[False, True]).iloc[0]
    cid = int(best.theme_id)
    sub = res["bt"].merged_df[res["bt"].merged_df["Topic"] == cid]
    if sub.empty:
        continue
    c = _as_centroid({"embedding": sub.iloc[0]["Embedding"]})
    print(f"\nEmbed parent T{pid} → best pre-CHAT sub T{cid}: {best.keywords[:60]}")
    for i, h in enumerate(rep_headlines(c, res["df"], res["emb"], n=3), 1):
        print(f"  rep {i}: {h[:95]}")

 parent_id  method  n_subthemes  pre_chat_stable  genai_keyword_hits                                                 top_pre_chat
        90 keyword           12               12                   0 damage, stimulus, checks, stimulus checks, damages, trump, j

Milestones: ChatGPT 2022-11-30 · CHAT 2023-05-17


## 7. Quick reload (no re-run)

Run §0–§2, then this cell.

In [8]:
cmp = OUTPUT_DIR / "genai_drilldown_compare.parquet"
if cmp.exists():
    print(pd.read_parquet(cmp).to_string(index=False))
for pat in sorted(OUTPUT_DIR.glob("scan_genai_embed_P*_*d.parquet")):
    t = pd.read_parquet(pat)
    pre = t[(t.slices >= MIN_ACTIVE_SLICES) & (pd.to_datetime(t.first_seen) < INCEPTION)]
    print(f"\n{pat.name}: {len(t)} subthemes, {len(pre)} pre-CHAT stable")
    for _, r in pre.head(5).iterrows():
        print(f"  T{int(r.theme_id):>3} {int(r.slices):>2}s  {pd.Timestamp(r.first_seen).date()}  {r.keywords[:50]}")

 parent_id  method  n_subthemes  pre_chat_stable  genai_keyword_hits                                                 top_pre_chat
        90 keyword           12               12                   0 damage, stimulus, checks, stimulus checks, damages, trump, j

scan_genai_embed_P90_21d.parquet: 5 subthemes, 5 pre-CHAT stable
  T  0 52s  2021-01-01  market, views, ahead, earnings, virus, economy, ma
  T  1 48s  2021-01-01  intel, chip, demand, forecast, estimates, revenue,
  T  4 44s  2021-01-01  chip shortage, chips, shortage, missing chips, car
  T  3 41s  2021-01-22  tech, taking, trade, market busy, busy, asian, pri
  T  2 41s  2021-01-01  techs, major techs, major, aussie, goes, dark, dar
